In [4]:
import pandas as pd
import numpy as np

chess_data = pd.read_csv("../dataset/chess_data.csv")
chess_data.head(50)

,move_number,player,from,to,piece,captured,promotion,san,time_sec,points_gained_this_move
0,1,White,h2,h4,p,NaN,NaN,h4,40.86,0
1,2,Black,e7,e5,p,NaN,NaN,e5,3.26,0
2,3,White,g1,f3,n,NaN,NaN,Nf3,11.68,0
3,4,Black,f8,a3,b,NaN,NaN,Ba3,3.72,0
4,5,White,b2,a3,p,b,NaN,bxa3,2.25,3
5,6,Black,e5,e4,p,NaN,NaN,e4,18.00,0
6,7,White,b1,c3,n,NaN,NaN,Nc3,4.26,0
7,8,Black,d7,d5,p,NaN,NaN,d5,11.00,0
8,9,White,c3,e4,n,p,NaN,Nxe4,2.35,1
9,10,Black,d5,e4,p,n,NaN,dxe4,4.37,3


In [6]:
chess_data.describe()
chess_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   move_number              10 non-null     int64  
 1   player                   10 non-null     object 
 2   from                     10 non-null     object 
 3   to                       10 non-null     object 
 4   piece                    10 non-null     object 
 5   captured                 3 non-null      object 
 6   promotion                0 non-null      float64
 7   san                      10 non-null     object 
 8   time_sec                 10 non-null     float64
 9   points_gained_this_move  10 non-null     int64  
dtypes: float64(2), int64(2), object(6)
memory usage: 932.0+ bytes


In [9]:
# Missing values
chess_data.isna().sum()

# Unique players
chess_data['player'].value_counts()


player
White    5
Black    5
Name: count, dtype: int64

In [ ]:
# Capture indicator
chess_data['is_capture'] = chess_data['captured'].notna()

# Fast move indicator
chess_data['fast_move'] = chess_data['time_sec'] < 5

# Game phase
chess_data['game_phase'] = np.where(
    chess_data['move_number'] <= 5, 'Early',
    np.where(chess_data['move_number'] <= 10, 'Mid', 'Late')
)

chess_data['player'].value_counts()

# Avg time per move
chess_data.groupby('player')['time_sec'].mean()

# Time by game phase
chess_data.groupby(['player', 'game_phase'])['time_sec'].mean()


player  game_phase
Black   Early          8.07
White   Early         12.28
Name: time_sec, dtype: float64

In [18]:
# Total captures
chess_data.groupby('player')['is_capture'].sum()

# Capture rate
chess_data.groupby('player')['is_capture'].mean()

# Total points gained
chess_data.groupby('player')['points_gained_this_move'].sum()

# Avg points per move
chess_data.groupby('player')['points_gained_this_move'].mean()

chess_data.groupby(['game_phase', 'player'])['points_gained_this_move'].sum()
chess_data.groupby(['player', 'piece']).size().unstack(fill_value=0)

chess_data.groupby(['player', 'piece']).size().unstack(fill_value=0)
# Fast moves vs material gain
chess_data.groupby('fast_move')['points_gained_this_move'].mean()

# Fast captures
chess_data[chess_data['fast_move'] & chess_data['is_capture']]


,move_number,player,from,to,piece,captured,promotion,san,time_sec,points_gained_this_move,is_capture,fast_move,game_phase
4,5,White,b2,a3,p,b,NaN,bxa3,2.25,3,True,True,Early
8,9,White,c3,e4,n,p,NaN,Nxe4,2.35,1,True,True,Early
9,10,Black,d5,e4,p,n,NaN,dxe4,4.37,3,True,True,Early


<!-- Player behavior differs more in time usage than move count

Material advantage is determined by few critical moves

Simple behavioral metrics are sufficient to extract meaningful insights from chess data -->

Most moves do not result in material gain; overall outcomes are driven by a small number of capture events.

Early game shows the highest concentration of captures and material exchanges, indicating that advantages are often established early.

Fast moves (low decision time) are frequently associated with capture events, suggesting reactive or forced responses rather than planned play.

Both players gain material in early exchanges, but few high-impact moves account for most of the total points gained.

Pawns and knights are involved in the majority of early captures, reflecting standard opening and development behavior.

Material gain is unevenly distributed across moves, reinforcing that single critical decisions can significantly influence game progression.